## 3. EDA Gold — Análise para Camada de Consumo (Silver → Gold)

### a) Imports e path

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pyspark.sql.functions as F

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(project_root)

### b) Carregando a base Silver

In [ ]:
%load_ext autoreload
%autoreload 2

from src.config.settings import Tables

silver_table = f"{Tables.SCHEMA}.{Tables.SILVER_TAXI}"
df_silver = spark.table(silver_table)

total = df_silver.count()
print(f"Tabela: {silver_table}")
print(f"Total de registros: {total:,}  |  Colunas: {len(df_silver.columns)}")
display(df_silver.limit(5))

### c) Completude das colunas exigidas pelo case

O case exige cinco colunas para responder às perguntas de negócio:
- `VendorID` — identificação do fornecedor TPEP
- `tpep_pickup_datetime` — âncora temporal das análises por mês e por hora
- `tpep_dropoff_datetime` — duração da corrida (contexto complementar)
- `passenger_count` — métrica principal de Q2
- `total_amount` — métrica principal de Q1

Antes de definir os filtros da Gold, é necessário medir a completude dessas colunas e quantificar
o impacto de cada valor problemático nas métricas finais.

In [ ]:
%load_ext autoreload
%autoreload 2

required_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "total_amount"]

null_exprs = [
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in required_cols
]

df_nulls = df_silver.agg(*null_exprs).toPandas().T
df_nulls.columns = ["nulls"]
df_nulls = df_nulls.reset_index()
df_nulls = df_nulls.rename(columns={"index": "coluna"})
df_nulls["pct_nulo"] = (df_nulls["nulls"] / total * 100).round(4)
df_nulls["registros_validos"] = total - df_nulls["nulls"]
df_nulls["pct_valido"] = (df_nulls["registros_validos"] / total * 100).round(4)

print("Completude das 5 colunas exigidas pelo case:")
display(df_nulls)

# Impacto adicional: valores semanticamente problemáticos
null_pass  = df_silver.filter(F.col("passenger_count").isNull()).count()
zero_pass  = df_silver.filter(F.col("passenger_count") == 0).count()
neg_total  = df_silver.filter(F.col("total_amount") <= 0).count()

print(f"\npassenger_count NULL  : {null_pass:,}  ({null_pass/total*100:.4f}%)  → distorce média de passageiros")
print(f"passenger_count == 0  : {zero_pass:,}  ({zero_pass/total*100:.4f}%)  → corrida sem passageiro registrado")
print(f"total_amount <= 0     : {neg_total:,}  ({neg_total/total*100:.4f}%)  → chargebacks + gratuidades")

### d) Análise Q1 — total_amount por mês

**Pergunta do case:** Qual foi o valor total recebido pelos motoristas de táxi amarelo em NYC por mês?

`total_amount` representa o valor final cobrado ao passageiro, incluindo todas as tarifas, impostos e gorjeta.
Valores negativos, confirmados na EDA Silver como chargebacks de `payment_type=4` (Dispute), representam
dinheiro **devolvido** ao passageiro — semanticamente o oposto de "valor recebido". \
A Gold deve filtrar `total_amount <= 0` para responder corretamente à pergunta.

In [ ]:
%load_ext autoreload
%autoreload 2

# Distribuição mensal — Silver completa (sem filtros)
print("Distribuição de total_amount por mês (Silver completa):")
display(
    df_silver
    .groupBy("month")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.sum("total_amount"), 2).alias("total_amount_sum"),
        F.round(F.avg("total_amount"), 4).alias("total_amount_avg"),
        F.round(F.min("total_amount"), 2).alias("min"),
        F.round(F.max("total_amount"), 2).alias("max"),
    )
    .orderBy("month")
)

# Impacto dos chargebacks na soma mensal
print("\nImpacto dos registros com total_amount <= 0 na soma mensal:")
display(
    df_silver
    .filter(F.col("total_amount") <= 0)
    .groupBy("month")
    .agg(
        F.count("*").alias("registros_negativos_ou_zero"),
        F.round(F.sum("total_amount"), 2).alias("valor_chargeback"),
    )
    .orderBy("month")
)

# Comparativo com vs sem chargebacks
df_q1_com    = df_silver.groupBy("month").agg(F.round(F.sum("total_amount"), 2).alias("total_com_chargeback"))
df_q1_sem    = df_silver.filter(F.col("total_amount") > 0).groupBy("month").agg(F.round(F.sum("total_amount"), 2).alias("total_sem_chargeback"))
df_q1_impact = df_q1_com.join(df_q1_sem, "month").orderBy("month")
df_q1_impact = df_q1_impact.withColumn("diferenca", F.round(F.col("total_sem_chargeback") - F.col("total_com_chargeback"), 2))

print("\nComparativo soma mensal com vs sem chargebacks:")
display(df_q1_impact)

### e) Análise Q2 — passenger_count por hora em Maio

**Pergunta do case:** Qual foi a quantidade média de passageiros por hora em Maio de 2023?

Dois tipos de registros distorcem a média:
- `passenger_count IS NULL`: falha do fornecedor TPEP (mesmos 428k registros com NULLs sistêmicos da EDA Silver). O Spark exclui NULLs automaticamente no `avg()`, mas o volume merece documentação.
- `passenger_count == 0`: reposicionamento de veículo ou falha de sensor — não é uma corrida com passageiro. Incluir zeros na média de passageiros por corrida seria semanticamente incorreto para a pergunta proposta.

##### ** A Gold filtrará `passenger_count IS NULL OR passenger_count <= 0` para Q2, e restringirá o mês 5 para esta análise. **

In [ ]:
%load_ext autoreload
%autoreload 2

df_may = df_silver.filter(F.col("month") == 5)
total_may = df_may.count()
print(f"Total de corridas em Maio: {total_may:,}")

# NULLs e zeros em Maio
null_may = df_may.filter(F.col("passenger_count").isNull()).count()
zero_may = df_may.filter(F.col("passenger_count") == 0).count()
valid_may = df_may.filter(F.col("passenger_count") > 0).count()

print(f"\npassenger_count NULL  : {null_may:,}  ({null_may/total_may*100:.4f}%)  → excluído do avg() automaticamente pelo Spark")
print(f"passenger_count == 0  : {zero_may:,}  ({zero_may/total_may*100:.4f}%)  → filtrado na Gold")
print(f"passenger_count > 0   : {valid_may:,}  ({valid_may/total_may*100:.4f}%)  → base para Q2")

# Distribuição por hora (passenger_count > 0, Maio)
print("\nMédia de passageiros por hora em Maio (passenger_count > 0):")
display(
    df_may
    .filter(F.col("passenger_count") > 0)
    .withColumn("hora", F.hour(F.col("tpep_pickup_datetime")))
    .groupBy("hora")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.avg("passenger_count"), 4).alias("avg_passageiros"),
        F.round(F.sum("passenger_count"), 0).alias("total_passageiros"),
    )
    .orderBy("hora")
)

# Comparativo avg com vs sem zeros
avg_com_zeros = df_may.filter(F.col("passenger_count").isNotNull()).agg(F.avg("passenger_count")).collect()[0][0]
avg_sem_zeros = df_may.filter(F.col("passenger_count") > 0).agg(F.avg("passenger_count")).collect()[0][0]

print(f"\nMédia global em Maio — com zeros    : {avg_com_zeros:.4f} passageiros/corrida")
print(f"Média global em Maio — sem zeros    : {avg_sem_zeros:.4f} passageiros/corrida")
print(f"Diferença (impacto dos zeros)       : {avg_com_zeros - avg_sem_zeros:.4f}")

### f) Distribuição dos campos de código numérico

`VendorID`, `RatecodeID` e `payment_type` são códigos numéricos cuja semântica está no dicionário da TLC. \
A Gold enriquecerá essas colunas com descrições legíveis via `CASE WHEN` para tornar a camada de consumo
auto-explicativa sem depender de joins externos. \
Esta seção valida a cobertura: confirma que todos os valores presentes na Silver estão mapeados no `CASE WHEN`.

In [ ]:
%load_ext autoreload
%autoreload 2

# Mapeamentos CASE WHEN que serão aplicados na Gold
vendor_desc_expr = (
    F.when(F.col("VendorID") == 1, "1 - Creative Mobile Technologies (CMT)")
    .when(F.col("VendorID") == 2, "2 - VeriFone Inc.")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("VendorID").cast("string")))
)

ratecode_desc_expr = (
    F.when(F.col("RatecodeID") == 1, "1 - Standard rate")
    .when(F.col("RatecodeID") == 2, "2 - JFK")
    .when(F.col("RatecodeID") == 3, "3 - Newark")
    .when(F.col("RatecodeID") == 4, "4 - Nassau or Westchester")
    .when(F.col("RatecodeID") == 5, "5 - Negotiated fare")
    .when(F.col("RatecodeID") == 6, "6 - Group ride")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("RatecodeID").cast("string")))
)

payment_desc_expr = (
    F.when(F.col("payment_type") == 0, "0 - Flex Fare trip")
    .when(F.col("payment_type") == 1, "1 - Credit card")
    .when(F.col("payment_type") == 2, "2 - Cash")
    .when(F.col("payment_type") == 3, "3 - No charge")
    .when(F.col("payment_type") == 4, "4 - Dispute")
    .when(F.col("payment_type") == 5, "5 - Unknown")
    .when(F.col("payment_type") == 6, "6 - Voided trip")
    .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("payment_type").cast("string")))
)

# VendorID
print("Distribuição VendorID:")
display(
    df_silver
    .withColumn("vendor_desc", vendor_desc_expr)
    .groupBy("VendorID", "vendor_desc")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.count("*") / total * 100, 4).alias("pct"),
    )
    .orderBy("VendorID")
)

# RatecodeID
print("Distribuição RatecodeID:")
display(
    df_silver
    .withColumn("ratecode_desc", ratecode_desc_expr)
    .groupBy("RatecodeID", "ratecode_desc")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.count("*") / total * 100, 4).alias("pct"),
    )
    .orderBy("RatecodeID")
)

# payment_type
print("Distribuição payment_type:")
display(
    df_silver
    .withColumn("payment_desc", payment_desc_expr)
    .groupBy("payment_type", "payment_desc")
    .agg(
        F.count("*").alias("corridas"),
        F.round(F.count("*") / total * 100, 4).alias("pct"),
    )
    .orderBy("payment_type")
)

### g) Sumário — filtros e enriquecimentos da camada Gold

| Decisão | Critério | Justificativa |
|---|---|---|
| Filtro Q1 | `total_amount > 0` | Chargebacks (`payment_type=4`) representam dinheiro **devolvido** — contradizem "valor recebido" |
| Filtro Q2 | `passenger_count > 0` | Zeros indicam reposicionamento/sensor — não representam corrida com passageiro |
| Filtro Q2 | `month == 5` | Pergunta restringe a Maio de 2023 |
| Projeção | Colunas exigidas pelo case + `year`/`month` de particionamento | Elimina colunas não necessárias para consumo analítico |
| Enriquecimento | `VendorID`, `RatecodeID`, `payment_type` → `_desc` via `CASE WHEN` | Camada de consumo auto-explicativa sem joins externos |
| NULLs sistêmicos | Nenhum filtro adicional | `avg()` do Spark ignora NULLs nativamente; registros ainda contribuem para Q1 com `total_amount` válido |

**Base Gold estimada:**
- Q1 (total_amount > 0): ~16.0M registros — ~99.1% da Silver
- Q2 (passenger_count > 0, month=5): ~3.4M registros — base de Maio menos zeros e NULLs

In [ ]:
%load_ext autoreload
%autoreload 2

from src.config.settings import Tables

# Colunas da Gold conforme requisito do case + enriquecimentos
GOLD_COLS = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "total_amount",
    "RatecodeID",
    "payment_type",
    "year",
    "month",
]

df_gold_preview = (
    df_silver
    .filter(F.col("total_amount") > 0)
    .select(*GOLD_COLS)
    .withColumn(
        "vendor_desc",
        F.when(F.col("VendorID") == 1, "1 - Creative Mobile Technologies (CMT)")
        .when(F.col("VendorID") == 2, "2 - VeriFone Inc.")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("VendorID").cast("string")))
    )
    .withColumn(
        "ratecode_desc",
        F.when(F.col("RatecodeID") == 1, "1 - Standard rate")
        .when(F.col("RatecodeID") == 2, "2 - JFK")
        .when(F.col("RatecodeID") == 3, "3 - Newark")
        .when(F.col("RatecodeID") == 4, "4 - Nassau or Westchester")
        .when(F.col("RatecodeID") == 5, "5 - Negotiated fare")
        .when(F.col("RatecodeID") == 6, "6 - Group ride")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("RatecodeID").cast("string")))
    )
    .withColumn(
        "payment_desc",
        F.when(F.col("payment_type") == 1, "1 - Credit card")
        .when(F.col("payment_type") == 2, "2 - Cash")
        .when(F.col("payment_type") == 3, "3 - No charge")
        .when(F.col("payment_type") == 4, "4 - Dispute")
        .when(F.col("payment_type") == 5, "5 - Unknown")
        .when(F.col("payment_type") == 6, "6 - Voided trip")
        .otherwise(F.concat(F.lit("Outro/Inválido: "), F.col("payment_type").cast("string")))
    )
)

gold_count = df_gold_preview.count()
print(f"Registros Gold (total_amount > 0): {gold_count:,}  ({gold_count/total*100:.2f}% da Silver)")
print(f"Colunas Gold: {len(df_gold_preview.columns)}")
print("\nPré-visualização da camada Gold:")
display(df_gold_preview.limit(10))